<a href="https://colab.research.google.com/github/Jopat2409/com3610_notebooks/blob/main/XLMR_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install torch
%pip install flair

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 65.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [ ]:
import torch
from flair.datasets import CONLL_03
from flair.embeddings import TransformerWordEmbeddings
from flair.models import SequenceTagger
from flair.trainers import ModelTrainer

# 1. get the corpus
corpus = CONLL_03(base_path=".")
print(corpus)

# 2. what label do we want to predict?
label_type = 'ner'

# 3. make the label dictionary from the corpus
label_dict = corpus.make_label_dictionary(label_type=label_type, add_unk=False)
print(label_dict)

# 4. initialize fine-tuneable transformer embeddings WITH document context
embeddings = TransformerWordEmbeddings(
    model='xlm-roberta-large',
    layers="-1",
    subtoken_pooling="first",
    fine_tune=True,
    use_context=True,
)

# 5. initialize bare-bones sequence tagger (no CRF, no RNN, no reprojection)
tagger = SequenceTagger(
    hidden_size=256,
    embeddings=embeddings,
    tag_dictionary=label_dict,
    tag_type='ner',
    use_crf=False,
    use_rnn=False,
    reproject_embeddings=False,
)

# 6. initialize trainer
trainer = ModelTrainer(tagger, corpus)

# 7. run fine-tuning
trainer.fine_tune(
    'resources/taggers/ner-english-large',
    learning_rate=5.0e-6,
    mini_batch_size=4,
    mini_batch_chunk_size=1,
    max_epochs=20,
    weight_decay=0.,
)

2025-03-04 20:10:29,050 Reading data from conll_03
2025-03-04 20:10:29,050 Train: conll_03/train.txt
2025-03-04 20:10:29,051 Dev: conll_03/dev.txt
2025-03-04 20:10:29,052 Test: conll_03/test.txt
Corpus: 14903 train + 3449 dev + 3658 test sentences
2025-03-04 20:10:38,903 Computing label dictionary. Progress:


1it [00:00, 1477.91it/s]
14903it [00:00, 33117.07it/s]

2025-03-04 20:10:39,359 Dictionary created for label 'ner' with 4 values: LOC (seen 8258 times), ORG (seen 7100 times), PER (seen 6527 times), MISC (seen 1681 times)


Dictionary with 4 tags: LOC, ORG, PER, MISC


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

...

2025-03-04 20:51:02,764 DEV : loss 0.10484579205513 - f1-score (micro avg)  0.8853
2025-03-04 20:51:02,834 ----------------------------------------------------------------------------------------------------


2025-03-04 20:54:48,670 epoch 2 - iter 372/3726 - loss 0.14151023 - time (sec): 225.83 - samples/sec: 92.16 - lr: 0.000003 - momentum: 0.000000
2025-03-04 20:58:32,488 epoch 2 - iter 744/3726 - loss 0.12055286 - time (sec): 449.65 - samples/sec: 91.70 - lr: 0.000003 - momentum: 0.000000
2025-03-04 21:02:13,418 ----------------------------------------------------------------------------------------------------
2025-03-04 21:02:13,419 Exiting from training early.
2025-03-04 21:02:13,420 Saving model ...
2025-03-04 21:02:34,148 Done.
2025-03-04 21:02:34,156 ----------------------------------------------------------------------------------------------------
2025-03-04 21:02:34,158 Testing using last state of model ...


100%|██████████| 229/229 [02:25<00:00,  1.57it/s]

2025-03-04 21:04:59,901 
Results:
- F-score (micro) 0.9143
- F-score (macro) 0.8638
- Accuracy 0.8766

By class:
              precision    recall  f1-score   support

         ORG     0.9479    0.8972    0.9219      1946
         LOC     0.9062    0.9580    0.9313      1784
         PER     0.9327    0.9749    0.9533      1591
        MISC     0.6372    0.6609    0.6488       404

   micro avg     0.9076    0.9210    0.9143      5725
   macro avg     0.8560    0.8727    0.8638      5725
weighted avg     0.9087    0.9210    0.9143      5725

2025-03-04 21:04:59,901 ----------------------------------------------------------------------------------------------------


{'test_score': 0.9142609449501518}